# Pandas Practice — Part 2 (No Solutions)

This notebook is a **pure practice** companion to `Pandas_for_ML_Part_2.ipynb`. It contains **no teaching content and no solutions** — just hands-on tasks so you can test yourself honestly.

## How to use this notebook
- Read each task carefully.
- Write your solution in the empty code cell provided right below it (or add a new cell).
- If you get stuck, re-open `Pandas_for_ML_Part_2.ipynb` and review the relevant section.
- Compare notes with a classmate or instructor once you're done — there's no answer key here on purpose.

Tasks are grouped into three levels:
- **Level 1 — Basic**: direct practice of a single Pandas concept.
- **Level 2 — Applied**: combine 2–3 Pandas concepts together.
- **Level 3 — ML-oriented Challenge**: realistic, data-flavored mini-tasks.

---
## Table of Contents
1. [Setup](#setup)
2. [Level 1 — Basic](#level1)
3. [Level 2 — Applied](#level2)
4. [Level 3 — ML-oriented Challenge](#level3)


<a id="setup"></a>
# Setup

Run this cell first — it creates the sample dataset used throughout this notebook's exercises.

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs("sample_data", exist_ok=True)

orders_csv_text = """order_id,customer,city,product,category,quantity,unit_price,order_date
1,Ali,Lahore,Laptop,Electronics,1,1200,2024-01-05
2,Sara,Karachi,Mouse,Electronics,3,25,2024-01-06
3,Ahmed,Lahore,Desk,Furniture,1,300,2024-01-15
4,Ayesha,Islamabad,Chair,Furniture,4,150,2024-01-20
5,Ali,Lahore,Monitor,Electronics,2,220,2024-02-02
6,Bilal,Karachi,Laptop,Electronics,1,1200,2024-02-10
7,Sara,Karachi,Desk,Furniture,2,300,2024-02-14
8,Zainab,Islamabad,Chair,Furniture,3,150,2024-02-25
9,Ahmed,Lahore,Monitor,Electronics,1,220,2024-03-01
10,Ayesha,Islamabad,Mouse,Electronics,5,25,2024-03-04
11,Ali,Lahore,Chair,Furniture,2,150,2024-03-08
12,Bilal,Karachi,Monitor,Electronics,2,220,2024-03-18
"""

with open("sample_data/orders.csv", "w") as file:
    file.write(orders_csv_text)

print("orders.csv is ready.")
print(pd.__version__)

<a id="level1"></a>
# Level 1 — Basic

### Task 1: Load and Add a Revenue Column
Read `sample_data/orders.csv` into a DataFrame called `orders`. Convert `order_date` to a proper datetime type. Add a `revenue` column equal to `quantity * unit_price`.

### Task 2: Group and Aggregate
Group `orders` by `category` and print the total `revenue` for each category.

### Task 3: Multiple Aggregations
Using `.agg()`, compute the total and average `revenue` per `city` in a single call.

### Task 4: String Cleaning
Add a column `product_upper` that stores each `product` name in uppercase, using the `.str` accessor.

### Task 5: Duplicate Check
Check whether `orders` has any exact duplicate rows. Then create a small DataFrame with at least one intentional duplicate row and demonstrate removing it with `.drop_duplicates()`.

In [ ]:
import pandas as pd
import numpy as np

# --- Task 1: Load and Add a Revenue Column ---
orders = pd.read_csv("sample_data/orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]

# --- Task 2: Group and Aggregate ---
category_revenue = orders.groupby("category")["revenue"].sum()
print("Total Revenue by Category:\n", category_revenue)

# --- Task 3: Multiple Aggregations ---
city_rev_stats = orders.groupby("city")["revenue"].agg(["sum", "mean"])
print("\nCity Revenue Stats (Total and Average):\n", city_rev_stats)

# --- Task 4: String Cleaning ---
orders["product_upper"] = orders["product"].str.upper()
print("\nProduct Upper Column Sample:\n", orders[["product", "product_upper"]].head())

# --- Task 5: Duplicate Check ---
print("\nExact duplicate rows in orders:", orders.duplicated().sum())

# Demonstrate drop_duplicates()
dup_df = pd.DataFrame({
    "id": [1, 2, 2, 3],
    "val": ["A", "B", "B", "C"]
})
print("\nBefore dropping duplicates:\n", dup_df)
print("After dropping duplicates:\n", dup_df.drop_duplicates())

<a id="level2"></a>
# Level 2 — Applied

### Task 6: Named Aggregation
Using `.groupby("city").agg(...)` with **named aggregation**, create a summary with columns `total_orders` (count of orders) and `total_revenue` (sum of revenue) per city.

### Task 7: Merge with Extra Data
Create a small DataFrame `city_region` mapping each city to a region:
```python
city_region = pd.DataFrame({
    "city": ["Lahore", "Karachi", "Islamabad"],
    "region": ["Punjab", "Sindh", "Federal"],
})
```
Merge it with `orders` (on `city`) so every order also has a `region` column. Use an appropriate `how` value and explain (in a comment) why you chose it.

### Task 8: Pivot Table
Build a pivot table showing total `revenue`, with `category` as rows and `city` as columns. Use `fill_value=0` for any missing combinations.

### Task 9: Apply with a Custom Function
Write a function that labels an order as `"Bulk"` if `quantity >= 3`, otherwise `"Regular"`. Use `.apply()` to create a new `order_type` column based on this function.

### Task 10: Dates
Extract the `order_month` and `order_weekday` (name, not number) from `order_date` into two new columns. Then find which weekday has the highest total revenue.

In [ ]:
# --- Task 6: Named Aggregation ---
city_summary = orders.groupby("city").agg(
    total_orders=("order_id", "count"),
    total_revenue=("revenue", "sum")
)
print("City Summary (Named Aggregation):\n", city_summary)

In [ ]:
# --- Task 7: Merge with Extra Data ---
city_region = pd.DataFrame({
    "city": ["Lahore", "Karachi", "Islamabad"],
    "region": ["Punjab", "Sindh", "Federal"],
})
# We use a left join (how='left') to keep all original orders intact 
# even if a city name was missing in our mapping table.
orders_merged = pd.merge(orders, city_region, on="city", how="left")
print("\nMerged Orders Sample:\n", orders_merged[["order_id", "city", "region"]].head())

In [ ]:
# --- Task 8: Pivot Table ---
pivot_rev = orders.pivot_table(
    index="category", 
    columns="city", 
    values="revenue", 
    aggfunc="sum", 
    fill_value=0
)
print("\nPivot Table (Revenue by Category and City):\n", pivot_rev)


In [ ]:
# --- Task 9: Apply with a Custom Function ---
def label_order_type(qty):
    return "Bulk" if qty >= 3 else "Regular"

orders["order_type"] = orders["quantity"].apply(label_order_type)
print("\nOrder Type Sample:\n", orders[["quantity", "order_type"]].head(4))


In [ ]:
# --- Task 10: Dates ---
orders["order_month"] = orders["order_date"].dt.to_period("M")
orders["order_weekday"] = orders["order_date"].dt.day_name()

weekday_revenue = orders.groupby("order_weekday")["revenue"].sum()
best_weekday = weekday_revenue.idxmax()
print(f"\nWeekday with highest total revenue: {best_weekday} ({weekday_revenue[best_weekday]})")


<a id="level3"></a>
# Level 3 — ML-oriented Challenge

### Task 11: Customer-Level Feature Engineering
Using `orders`, build a customer-level summary DataFrame with one row per `customer`, containing:
- `total_orders` — number of orders placed
- `total_spent` — total revenue across all their orders
- `average_order_value` — their average revenue per order
- `favorite_category` — the category they ordered the most (hint: this is the trickiest part — think about grouping by both `customer` and `category`, counting, then finding each customer's top category)

### Task 12: Monthly Trend and Growth
Compute total revenue per `order_month`. Then compute the **month-over-month change** in revenue (this month's total minus the previous month's total) as a new Series. (Hint: look into whether a Pandas Series method can compute the difference between consecutive values without a manual loop.)

### Task 13: End-to-End Mini Report
Combine everything you've practiced to produce a single, clean summary DataFrame with one row per `city`, containing: `total_revenue`, `total_orders`, `average_order_value`, and `top_category` (the category with the most revenue in that city). Export this summary to both a CSV file and a JSON file (`orient="records"`), then reload each one and print it to confirm it round-trips correctly.

In [ ]:
# --- Task 11: Customer-Level Feature Engineering ---
# Step A: Aggregate basic stats per customer
cust_summary = orders.groupby("customer").agg(
    total_orders=("order_id", "count"),
    total_spent=("revenue", "sum"),
    average_order_value=("revenue", "mean")
)

# Step B: Find favorite category per customer
# Group by customer and category, count orders, reset index, sort, and drop duplicates keeping the first
cat_counts = orders.groupby(["customer", "category"]).size().reset_index(name="count")
cat_counts = cat_counts.sort_values(by=["customer", "count"], ascending=[True, False])
favorite_categories = cat_counts.drop_duplicates(subset=["customer"]).set_index("customer")["category"]

# Step C: Combine into final customer-level DataFrame
cust_summary["favorite_category"] = favorite_categories
print("Customer-Level Summary:\n", cust_summary)


In [ ]:
# --- Task 12: Monthly Trend and Growth ---
monthly_rev = orders.groupby("order_month")["revenue"].sum()
mom_change = monthly_rev.diff()
print("\nMonthly Revenue Trend:\n", monthly_rev)
print("\nMonth-over-Month Revenue Change:\n", mom_change)


In [ ]:
# --- Task 13: End-to-End Mini Report ---
# Step A: City-level aggregation metrics
city_report = orders.groupby("city").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count"),
    average_order_value=("revenue", "mean")
)

# Step B: Find top category per city
city_cat = orders.groupby(["city", "category"])["revenue"].sum().reset_index()
city_cat = city_cat.sort_values(by=["city", "revenue"], ascending=[True, False])
top_cats = city_cat.drop_duplicates(subset=["city"]).set_index("city")["category"]
city_report["top_category"] = top_cats

print("\nFinal End-to-End City Report:\n", city_report)

# Step C: Export and round-trip confirmation
city_report.reset_index().to_csv("sample_data/city_report.csv", index=False)
city_report.reset_index().to_json("sample_data/city_report.json", orient="records")

loaded_csv = pd.read_csv("sample_data/city_report.csv")
loaded_json = pd.read_json("sample_data/city_report.json")

print("\nReloaded from CSV:\n", loaded_csv)
print("\nReloaded from JSON:\n", loaded_json)


---
That's the end of the Pandas practice series. Once you're comfortable with everything here, you're ready to move on to **Data Visualization** and then **Scikit-learn**, where this cleaned, structured data becomes the direct input to real Machine Learning models.